# Suite2p Video Notebook

## Imports and Config

In [5]:
import os

import cv2
import h5py
import numpy as np

movie_path = os.path.join(
    "libs",
    "ciatah",
    "data",
    "twoPhoton",
    "2017_04_16_p485_m487_runningWheel02",
    "concat_recording_20140807_102507.h5",
)
h5_key = "1"
traces_path = os.path.join("data", "suite2p_traces", "suite2p_dff_traces.csv")
cascade_path = os.path.join("data", "spike_detection", "cascade_spike_rates.npy")
oasis_path = os.path.join("data", "spike_detection", "oasis_spikes.npy")
suite2p_selected_rois_path = os.path.join("data", "suite2p_traces", "suite2p_selected_rois.csv")
suite2p_stat_path = os.path.join("data", "suite2p_output", "suite2p", "plane0", "stat.npy")
video_path = os.path.join("videos", "suite2p_cascade_oasis_realtime.mp4")

trace_index = 2
frame_rate = 30.0
output_fps = 30.0
start_s = 0.0
duration_s = 30.0
window_s = 30.0

## Load traces and selected neuron

In [6]:
traces = np.loadtxt(traces_path, delimiter=",").astype(np.float32)
cascade = np.load(cascade_path).astype(np.float32)
oasis = np.load(oasis_path).astype(np.float32)

selected_rois = np.loadtxt(suite2p_selected_rois_path, delimiter=",", skiprows=1)
selected_rois = np.atleast_2d(selected_rois)
suite2p_roi_index = int(selected_rois[trace_index, 0])
stat = np.load(suite2p_stat_path, allow_pickle=True)[suite2p_roi_index]

print("trace shape:", traces.shape)
print("cascade shape:", cascade.shape)
print("oasis shape:", oasis.shape)
print("trace index:", trace_index)
print("suite2p roi index:", suite2p_roi_index)

trace shape: (39, 3000)
cascade shape: (39, 3000)
oasis shape: (39, 3000)
trace index: 2
suite2p roi index: 2


## Video helpers

In [7]:
def normalize_image(frame, low, high):
    frame = (frame.astype(np.float32) - low) / (high - low)
    frame = np.clip(frame, 0, 1)
    return (frame * 255).astype(np.uint8)

def normalize_trace(values):
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    low, high = np.nanpercentile(values[finite], [1, 99])
    if high <= low:
        return np.zeros_like(values)
    values = np.clip((values - low) / (high - low), 0, 1)
    return np.nan_to_num(values, nan=0.0, posinf=1.0, neginf=0.0)

def draw_text(image, text, origin, scale=0.65, color=(245, 245, 245), thickness=1):
    cv2.putText(
        image,
        text,
        origin,
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        color,
        thickness,
        cv2.LINE_AA,
    )

def draw_roi_outline(panel, stat, image_shape):
    ypix = np.asarray(stat["ypix"], dtype=np.int32)
    xpix = np.asarray(stat["xpix"], dtype=np.int32)
    mask = np.zeros(image_shape, dtype=np.uint8)
    yy = np.clip(ypix, 0, mask.shape[0] - 1)
    xx = np.clip(xpix, 0, mask.shape[1] - 1)
    mask[yy, xx] = 255
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    scale_y = panel.shape[0] / image_shape[0]
    scale_x = panel.shape[1] / image_shape[1]
    scaled_contours = []
    for contour in contours:
        contour = contour.astype(np.float32)
        contour[:, :, 0] *= scale_x
        contour[:, :, 1] *= scale_y
        scaled_contours.append(contour.astype(np.int32))

    cv2.drawContours(panel, scaled_contours, -1, (0, 240, 255), 2)

def draw_trace_panel(panel, trace_data, frame_idx, frame_rate, window_s):
    panel[:] = (250, 250, 250)
    height, width = panel.shape[:2]
    margin_left = 70
    margin_right = 24
    margin_top = 54
    margin_bottom = 54
    plot_x0 = margin_left
    plot_y0 = margin_top
    plot_x1 = width - margin_right
    plot_y1 = height - margin_bottom

    cv2.rectangle(panel, (plot_x0, plot_y0), (plot_x1, plot_y1), (40, 40, 40), 1)

    window_frames = max(2, int(round(window_s * frame_rate)))
    start = max(0, frame_idx - window_frames + 1)
    end = frame_idx + 1
    local_frames = np.arange(start, end)

    colors = {
        "RAW": (120, 120, 120),
        "CASCADE": (170, 90, 30),
        "OASIS": (95, 150, 35),
    }

    y_axis_max = 1.25
    for name, values in trace_data:
        yvals = values[start:end]
        x = plot_x0 + ((local_frames - start) / max(1, window_frames - 1)) * (plot_x1 - plot_x0)
        y = plot_y1 - (yvals / y_axis_max) * (plot_y1 - plot_y0)
        y = np.clip(y, plot_y0 + 2, plot_y1 - 2)
        points = np.column_stack((x, y)).astype(np.int32)
        if points.shape[0] > 1:
            cv2.polylines(panel, [points], False, colors[name], 1, cv2.LINE_AA)

    cursor_x = plot_x0 + int((plot_x1 - plot_x0) * (end - start - 1) / max(1, window_frames - 1))
    cv2.line(panel, (cursor_x, plot_y0), (cursor_x, plot_y1), (20, 20, 20), 1)

    current_time = frame_idx / frame_rate
    draw_text(panel, "Spike detection", (plot_x0, 32), 0.72, (30, 30, 30), 2)
    draw_text(panel, f"t = {current_time:05.2f} s", (plot_x0, height - 18), 0.62, (30, 30, 30), 1)
    draw_text(panel, "0", (22, plot_y1 + 4), 0.55, (40, 40, 40), 1)
    draw_text(panel, "1", (22, plot_y0 + 5), 0.55, (40, 40, 40), 1)

    legend_x = plot_x1 - 210
    legend_y = plot_y0 + 34
    cv2.rectangle(panel, (legend_x - 14, legend_y - 22), (plot_x1 - 12, legend_y + 62), (250, 250, 250), -1)
    cv2.rectangle(panel, (legend_x - 14, legend_y - 22), (plot_x1 - 12, legend_y + 62), (90, 90, 90), 1)

    for idx, name in enumerate(["RAW", "CASCADE", "OASIS"]):
        y = legend_y + idx * 24
        cv2.line(panel, (legend_x, y), (legend_x + 34, y), colors[name], 2)
        draw_text(panel, name, (legend_x + 44, y + 6), 0.58, (30, 30, 30), 1)

## Build and save video

In [8]:
trace_data = [
    ("RAW", normalize_trace(traces[trace_index])),
    ("CASCADE", normalize_trace(cascade[trace_index])),
    ("OASIS", normalize_trace(oasis[trace_index])),
]

os.makedirs(os.path.dirname(video_path), exist_ok=True)
preview_path = os.path.splitext(video_path)[0] + "_preview.png"

with h5py.File(movie_path, "r") as movie_file:
    movie = movie_file[h5_key]
    n_frames = min(movie.shape[0], traces.shape[1], cascade.shape[1], oasis.shape[1])
    start_frame = max(0, int(round(start_s * frame_rate)))
    frames_to_write = min(n_frames - start_frame, int(round(duration_s * frame_rate)))

    sample_end = min(n_frames, start_frame + max(frames_to_write, 300))
    sample = movie[start_frame:sample_end]
    low, high = np.percentile(sample, [1, 99.8])
    if high <= low:
        high = low + 1

    movie_panel_size = 520
    trace_panel_width = 920
    output_height = movie_panel_size
    output_width = movie_panel_size + trace_panel_width

    writer = cv2.VideoWriter(
        video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        output_fps,
        (output_width, output_height),
    )

    preview_frame = None
    for output_idx in range(frames_to_write):
        frame_idx = start_frame + output_idx
        frame = normalize_image(movie[frame_idx], low, high)
        movie_panel = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        movie_panel = cv2.resize(movie_panel, (movie_panel_size, movie_panel_size))
        draw_roi_outline(movie_panel, stat, frame.shape)
        draw_text(movie_panel, "Two-photon calcium movie", (16, 32), 0.7, (255, 255, 255), 2)

        trace_panel = np.zeros((output_height, trace_panel_width, 3), dtype=np.uint8)
        draw_trace_panel(trace_panel, trace_data, frame_idx, frame_rate, window_s)

        combined = np.concatenate((movie_panel, trace_panel), axis=1)
        writer.write(combined)

        if output_idx == frames_to_write // 2:
            preview_frame = combined.copy()

    writer.release()

if preview_frame is not None:
    cv2.imwrite(preview_path, preview_frame)

print("saved video:", video_path)
print("saved preview:", preview_path)
print("frames written:", frames_to_write)

saved video: videos/suite2p_cascade_oasis_realtime.mp4
saved preview: videos/suite2p_cascade_oasis_realtime_preview.png
frames written: 900
